# Simple Handoffs

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to use handoffs to route conversations between agents — the triage pattern, checking which agent answered, and inspecting handoff traces.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. The Triage Pattern

Define specialist agents and a triage agent that routes to them. Each agent in `handoffs` becomes a `transfer_to_<name>` tool.

In [ ]:
from agents import Agent, Runner

math_agent = Agent(
    name="Math Tutor",
    instructions="You are a math tutor. Explain math concepts clearly with step-by-step solutions.",
)

history_agent = Agent(
    name="History Expert",
    instructions="You are a history expert. Provide detailed, accurate historical information.",
)

triage = Agent(
    name="Triage Agent",
    instructions="""You are a routing agent. Based on the user's question:
- Hand off to Math Tutor for math questions
- Hand off to History Expert for history questions
- For other topics, answer directly.""",
    handoffs=[math_agent, history_agent],
)

result = Runner.run_sync(triage, "What caused the French Revolution?")
print(result.final_output)

## 4. Checking Which Agent Answered

Use `result.last_agent` to see which agent produced the final output.

In [ ]:
result = Runner.run_sync(triage, "What is the integral of x squared?")

print(f"Answer: {result.final_output[:200]}...")
print(f"\nAnswered by: {result.last_agent.name}")

## 5. Testing Multiple Routes

Send different types of questions and confirm they route to the correct specialist.

In [ ]:
questions = [
    "Solve for x: 2x + 5 = 13",
    "Who built the Great Wall of China?",
    "What is the Pythagorean theorem?",
    "When did World War 1 start?",
]

for q in questions:
    result = Runner.run_sync(triage, q)
    print(f"Q: {q}")
    print(f"Routed to: {result.last_agent.name}")
    print("-" * 40)

## 6. Inspecting the Handoff Trace

The `result.new_items` list contains every message and tool call, including the handoff event.

In [ ]:
result = Runner.run_sync(triage, "When did the Roman Empire fall?")

print(f"Final agent: {result.last_agent.name}")
print(f"\nFull trace ({len(result.new_items)} items):")
for item in result.new_items:
    print(item)

## 7. Chained Handoffs

Handoffs can be chained — a specialist can hand off to another specialist for even finer routing.

In [ ]:
algebra_agent = Agent(
    name="Algebra Specialist",
    instructions="You solve algebra problems step by step.",
)

calculus_agent = Agent(
    name="Calculus Specialist",
    instructions="You solve calculus problems step by step.",
)

math_router = Agent(
    name="Math Router",
    instructions="""Route math questions to the right specialist:
- Algebra Specialist for equations, variables, polynomials
- Calculus Specialist for derivatives, integrals, limits""",
    handoffs=[algebra_agent, calculus_agent],
)

main_triage = Agent(
    name="Main Triage",
    instructions="Route to Math Router for any math questions. Answer other questions directly.",
    handoffs=[math_router],
)

result = Runner.run_sync(main_triage, "Find the derivative of x^3 + 2x")
print(f"Answer: {result.final_output[:200]}...")
print(f"Answered by: {result.last_agent.name}")

## Key Takeaways

- Handoffs let agents transfer control to specialists via `Agent(handoffs=[...])`
- Each handoff agent becomes a `transfer_to_<name>` tool the agent can call
- `result.last_agent` tells you which agent produced the final output
- Handoffs can be chained for multi-level routing
- The full handoff trace is available in `result.new_items`